# Phase 5: Gaussian Hidden Markov Model (HMM) Training

**Objective**: Formulate and train a Gaussian HMM on temporal driving sequences:
- **Hidden states $S = \{s_1, \dots, s_K\}$**: Underlying driving conditions (e.g. stable cruising, harsh acceleration, hard braking, sharp turn)
- **Observations $O_t \in \mathbb{R}^D$**: Windowed kinematic feature vectors
- **Initial distribution $\pi$**: $P(q_1 = s_i)$
- **Transition matrix $A$**: $A_{ij} = P(q_{t+1} = s_j \mid q_t = s_i)$
- **Emission distributions $B$**: $b_i(O_t) = \mathcal{N}(O_t; \mu_i, \Sigma_i)$

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.sequences import build_sequences_from_features, SequenceNormaliser, split_sequences_chronologically
from src.hmm_model import train_gaussian_hmm, build_state_label_map, sweep_n_states, save_model

feat_df = pd.read_csv('../data/processed/features_all.csv')
sequences, lengths, meta = build_sequences_from_features(feat_df)
(train_seqs, val_seqs, test_seqs,
 train_meta, val_meta, test_meta) = split_sequences_chronologically(sequences, meta, train_frac=0.7, val_frac=0.15, gap=1)

normaliser = SequenceNormaliser()
train_norm = normaliser.fit_transform(train_seqs)
val_norm = normaliser.transform(val_seqs)

## 1. Hyperparameter Sweep: Number of Hidden States $K \in \{3, 4, 5\}$

In [ ]:
val_labels = [m['label'] for m in val_meta]
sweep_df = sweep_n_states(train_norm, val_norm, val_labels, n_states_list=[3, 4, 5])
display(sweep_df)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].plot(sweep_df['n_states'], sweep_df['accuracy']*100, 'o-', color='#2980b9')
axes[0].set_title("Validation Accuracy (%)", fontweight="bold")
axes[0].set_xlabel("K States")

axes[1].plot(sweep_df['n_states'], sweep_df['aic'], 's-', color='#e74c3c')
axes[1].set_title("AIC (Lower is Better)", fontweight="bold")
axes[1].set_xlabel("K States")

axes[2].plot(sweep_df['n_states'], sweep_df['bic'], '^-', color='#27ae60')
axes[2].set_title("BIC (Lower is Better)", fontweight="bold")
axes[2].set_xlabel("K States")
plt.tight_layout()
plt.show()

## 2. Train Chosen Gaussian HMM (K=4)

In [ ]:
model = train_gaussian_hmm(train_norm, n_states=4, covariance_type='diag', n_iter=100, random_state=42)
train_labels = [m['label'] for m in train_meta]
state_map = build_state_label_map(model, train_norm, train_labels)

print("Learned Transition Matrix A:")
print(np.round(model.transmat_, 3))
print("\nState to Behavior Category Mapping:")
print(state_map)